# OCR em imagens novas

Este caderno roda a **configuração final** (sem ajustar nada) num lote de imagens novas, no mesmo formato das páginas do livro, e:

1. grava um `.txt` por imagem em `SAIDA/texts/<nome da imagem>.txt`;
2. mostra o processamento completo de até `MOSTRAR` páginas: cada operação de imagem nos dois caminhos (segmentação e reconhecimento), as linhas encontradas, o texto depois de cada etapa e, linha a linha, o recorte da imagem ao lado do texto lido;
3. se houver transcrições de referência, calcula WER e CER por página e do lote.

**Entradas**
- `IMAGENS`: pasta com as imagens (`.jpg`, `.jpeg`, `.png`, `.tif`).
- `REFERENCIAS` (opcional): pasta com as transcrições, um `.txt` com **o mesmo nome da imagem** (`pagina_01.jpg` ↔ `pagina_01.txt`). Imagens sem referência são transcritas, mas ficam fora do WER/CER. Use `None` se não houver nenhuma.
- `CONFIG`: a configuração escolhida pela busca, só com as páginas de ajuste.

A lógica está em `ocr_pipeline/batch.py`, que também roda pela linha de comando:
`python -m ocr_pipeline.batch --config configs/final_v3.json --images <pasta> [--references <pasta>] --out outputs/batch`.

## Ambiente

In [ ]:
# No Colab: clona o repositório, entra nele e instala as dependências (o torch do Colab é mantido).
# Localmente, abra o caderno a partir da raiz do repositório; esta célula não faz nada.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORIO = 'https://github.com/fabiolobato/hackathon_ocr_historico.git'
RAMO = 'feat/random_search_period_lexicon'

if 'google.colab' in sys.modules:
    clone = Path('/content/hackathon_ocr_historico')
    if not clone.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '-b', RAMO, REPOSITORIO, str(clone)], check=True)
    os.chdir(clone)
    subprocess.run([sys.executable, 'docs/entrega/setup_pipeline.py'], check=True)  # Kraken, modelo PP-OCRv6
elif not Path('ocr_pipeline').is_dir() and Path('../../ocr_pipeline').is_dir():
    os.chdir('../..')  # aberto de dentro de docs/entrega
sys.path.insert(0, str(Path.cwd()))  # o pacote ocr_pipeline está na raiz do repositório
print('pasta de trabalho:', Path.cwd())

## Parâmetros

Os dicionários de época (`models/lexicon/...`) não são baixados pelo `setup_pipeline.py`. Gere-os com `tools/build_period_lexicon.py` (e `tools/fetch_alesp_acts.py`, se a configuração usar o corpus legislativo) ou copie a pasta `models/lexicon`. Se faltar algum arquivo, a próxima célula avisa qual.

In [ ]:
import os
import sys

import torch

CONFIG = Path('configs/final_v3.json')
IMAGENS = Path('novas_imagens')        # pasta com as imagens novas
RECORTAR = False                       # True: imagens inteiras do acervo FPMSC (Flickr), sem o recorte das 12 páginas;
#                                        tira a faixa da marca d'água e as bordas (ocr_pipeline/page_crop.py)
REFERENCIAS = Path('novas_referencias')  # pasta com <nome da imagem>.txt, ou None
SAIDA = Path('outputs/novas_imagens')  # textos, resumo e cache (o cache fica aqui para não misturar com as 12 páginas)
MOSTRAR = 5                            # quantas páginas exibir com o processamento completo
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
# páginas processadas em paralelo: no Colab, todos os núcleos; localmente, metade (o computador segue usável)
TRABALHADORES = os.cpu_count() if 'google.colab' in sys.modules else max(1, (os.cpu_count() or 2) // 2)

In [ ]:
import json
import logging
import warnings

from IPython.display import HTML, display

from ocr_pipeline.batch import card_html, check_config, discover_batch, run_batch
from ocr_pipeline.config import load_config
from ocr_pipeline.engine import Engine
from ocr_pipeline.metrics import corpus
from ocr_pipeline.page_crop import crop_folder
from ocr_pipeline.pipeline import page_executor
from ocr_pipeline.presentation import style

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings('ignore', module='coremltools')  # aviso inofensivo do Kraken no macOS

config = load_config(CONFIG)
check_config(config)
referencias = REFERENCIAS if REFERENCIAS is not None and Path(REFERENCIAS).is_dir() else None
if RECORTAR:
    crop_folder(IMAGENS, SAIDA / 'recortadas')
    IMAGENS = SAIDA / 'recortadas'
paginas = discover_batch(IMAGENS, referencias)
engine = Engine(SAIDA / 'cache', DISPOSITIVO, 42)
print(f'{len(paginas)} imagens, {sum(p.reference is not None for p in paginas)} com referência')
print('configuração:', CONFIG)
print(f'dispositivo: {DISPOSITIVO}, {TRABALHADORES} página(s) em paralelo')

## Transcrição do lote

A primeira execução leva de 20 a 60 s por página em CPU. Segmentação e reconhecimento ficam em `SAIDA/cache`, então rodar de novo é imediato.

In [ ]:
if TRABALHADORES > 1:
    with page_executor(engine, TRABALHADORES) as executor:
        linhas, tracos = run_batch(engine, paginas, config, SAIDA, executor)
else:
    linhas, tracos = run_batch(engine, paginas, config, SAIDA)
for linha in linhas:
    print(f"{linha['image']:40s} {linha['lines']:3d} linhas {linha['words']:4d} palavras  -> {linha['text_file']}")

## Processamento completo (até `MOSTRAR` páginas)

Como ler cada cartão:
- **geometria / imagem de segmentação / imagem de reconhecimento**: cada imagem é a anterior com mais uma operação. A última da segmentação mostra as linhas encontradas (polígono + linha de base).
- **texto**: o texto depois de cada etapa (reconhecedor → reescore com dicionário e contexto → convenções → hifenização → correção por edição). Com referência, aparece o WER/CER de cada etapa e o texto final com as palavras erradas riscadas.
- **linha a linha**: o recorte de cada linha ao lado do que foi lido nela.

In [ ]:
display(HTML(style()))
for pagina in paginas[:MOSTRAR]:
    display(HTML(f'<div class="ocr-body">{card_html(pagina, tracos[pagina.number], config)}</div>'))

## Resumo do lote

WER e CER normalizados (minúsculas, sem pontuação, acentos mantidos) e estritos. O total do lote soma os erros de todas as páginas com referência e divide pelo total de palavras/caracteres; não é a média das páginas.

In [ ]:
com_referencia = [l for l in linhas if 'wer_norm' in l]
if not com_referencia:
    print('nenhuma imagem com referência: só os textos foram gerados')
else:
    print(f"{'imagem':40s} {'WER norm':>9s} {'CER norm':>9s} {'WER estrito':>12s}")
    for l in com_referencia:
        print(f"{l['image']:40s} {100 * l['wer_norm']:9.2f} {100 * l['cer_norm']:9.2f} {100 * l['wer_strict']:12.2f}")
    total = corpus(com_referencia)
    print(f"{'LOTE (' + str(len(com_referencia)) + ' páginas)':40s} {100 * total['wer_norm']:9.2f} "
          f"{100 * total['cer_norm']:9.2f} {100 * total['wer_strict']:12.2f}")
print('resumo completo em', SAIDA / 'summary.json')